In [ ]:
import pandas as pd
import numpy as np
import json
import time
import joblib
import os
from utils.neural_network import NeuralNetwork
from utils.metrics import *

# Best hyperparameters
BEST_HYPERPARAMETERS = {
    'sgd':       {'learning_rate': 0.01, 'momentum': 0.0},
    'adam':      {'learning_rate': 0.0001},
    'adabelief': {'learning_rate': 0.0001}
}

BEST_ARCHITECTURE       = [30, 128, 64, 32, 1]
BEST_BATCH_SIZE         = 64
MAX_EPOCHS              = 200
EARLY_STOPPING_PATIENCE = 15
EVALUATION_THRESHOLD    = 0.5
RANDOM_STATE            = 42

# Load datasets
df_train = pd.read_csv('dataset/creditcard_train.csv')
df_val   = pd.read_csv('dataset/creditcard_val.csv')
df_test  = pd.read_csv('dataset/creditcard_test.csv')

y_train = df_train['Class'];  X_train = df_train.drop('Class', axis=1)
y_val   = df_val['Class'];    X_val   = df_val.drop('Class', axis=1)
y_test  = df_test['Class'];   X_test  = df_test.drop('Class', axis=1)

print(f"Train: {X_train.shape} | Fraud: {y_train.sum()} ({y_train.mean()*100:.4f}%)")
print(f"Val:   {X_val.shape}   | Fraud: {y_val.sum()} ({y_val.mean()*100:.4f}%)")
print(f"Test:  {X_test.shape}  | Fraud: {y_test.sum()} ({y_test.mean()*100:.4f}%)")
print("\n✓ No SMOTE — menggunakan data asli (imbalanced)\n")

final_results = {}

for optimizer in ['sgd', 'adam', 'adabelief']:
    print("="*60)
    print(f"  Training {optimizer.upper()} (No SMOTE)")
    print("="*60)
    print(f"  Hyperparameters: {BEST_HYPERPARAMETERS[optimizer]}")

    start_time = time.time()

    nn = NeuralNetwork(
        layers=BEST_ARCHITECTURE,
        activation='relu',
        output_activation='sigmoid',
        random_state=RANDOM_STATE,
        momentum=BEST_HYPERPARAMETERS['sgd']['momentum']
    )

    nn.fit(
        X_train, y_train,
        X_val=X_val, y_val=y_val,
        epochs=MAX_EPOCHS,
        batch_size=BEST_BATCH_SIZE,
        learning_rate=BEST_HYPERPARAMETERS[optimizer]['learning_rate'],
        optimizer=optimizer,
        verbose=1,
        early_stopping_patience=EARLY_STOPPING_PATIENCE
    )

    training_time = time.time() - start_time
    epochs_trained = len(nn.history['loss'])

    # ── Evaluate on test set ──────────────────────────────
    y_pred_proba = nn.predict_proba(X_test).ravel()
    y_pred       = (y_pred_proba >= EVALUATION_THRESHOLD).astype(int)

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    metrics = {
        'pr_auc':    float(average_precision_score(y_test, y_pred_proba)),
        'roc_auc':   float(roc_auc_score(y_test, y_pred_proba)),
        'f1':        float(f1_score(y_test, y_pred, zero_division=0)),
        'precision': float(precision_score(y_test, y_pred, zero_division=0)),
        'recall':    float(recall_score(y_test, y_pred, zero_division=0)),
        'accuracy':  float(accuracy_score(y_test, y_pred)),
        'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn),
        'epochs': epochs_trained,
        'time_s': round(training_time, 2)
    }

    final_results[optimizer] = metrics

    print(f"\n  ✓ Done in {training_time:.1f}s ({epochs_trained} epochs)")
    print(f"    PR-AUC:  {metrics['pr_auc']:.4f}")
    print(f"    ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"    F1:      {metrics['f1']:.4f}")
    print(f"    Precision: {metrics['precision']:.4f} | Recall: {metrics['recall']:.4f}")
    print(f"    TP: {tp}  FP: {fp}  FN: {fn}  TN: {tn}")

    # ── Simpan model ──────────────────────────────────────
    model_path = f'saved/best_model_{optimizer}.pkl'
    joblib.dump(nn, model_path)
    print(f"  ✓ Model disimpan → {model_path}")

# ── Simpan semua metrics ke JSON dan CSV ──────────────────
with open('results/final_no_smote_metrics.json', 'w') as f:
    json.dump(final_results, f, indent=4)

print("\n" + "="*60)
print("  FINAL RESULTS — NO SMOTE")
print("="*60)
print("\n✓ Saved: saved/best_model_sgd.pkl")
print("✓ Saved: saved/best_model_adam.pkl")
print("✓ Saved: saved/best_model_adabelief.pkl")

Train: (198610, 30) | Fraud: 332 (0.1672%)
Val:   (42559, 30)   | Fraud: 71 (0.1668%)
Test:  (42557, 30)  | Fraud: 70 (0.1645%)

✓ No SMOTE — menggunakan data asli (imbalanced)

  Training SGD (No SMOTE)
  Hyperparameters: {'learning_rate': 0.01, 'momentum': 0.0}

Early stopping at epoch 80

  ✓ Done in 68.0s (80 epochs)
    PR-AUC:  0.8086
    ROC-AUC: 0.9558
    F1:      0.8154
    Precision: 0.8833 | Recall: 0.7571
    TP: 53  FP: 7  FN: 17  TN: 42480
  ✓ Model disimpan → saved/best_model_sgd.pkl
  Training ADAM (No SMOTE)
  Hyperparameters: {'learning_rate': 0.0001}

Early stopping at epoch 25

  ✓ Done in 28.7s (25 epochs)
    PR-AUC:  0.7980
    ROC-AUC: 0.9655
    F1:      0.7970
    Precision: 0.8413 | Recall: 0.7571
    TP: 53  FP: 10  FN: 17  TN: 42477
  ✓ Model disimpan → saved/best_model_adam.pkl
  Training ADABELIEF (No SMOTE)
  Hyperparameters: {'learning_rate': 0.0001}

Early stopping at epoch 27

  ✓ Done in 32.3s (27 epochs)
    PR-AUC:  0.8083
    ROC-AUC: 0.9624
    